In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install "pandas<2.0.0"
!pip install scikit-learn==1.2.2
!pip install mlxtend==0.23.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
xarray 2025.1.2 requires pandas>=2.1, but you have pandas 1.5.3 which is incompatible.
dask-cudf-cu12 24.12.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 1.5.3 which is incompatible.
cudf-cu12 24.12.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 1.5.3 which is incompatible.
mizani 0.13.1 requires pandas>=2.2.0, but you have pandas 1.5.3 which is incompatible.
dask-expr 1.1.19 requires pandas>=2, but you have pandas 1.5.3 which is incompatible.
plotnine 0.14.5 requires pandas>=2.2.

In [ ]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import itertools
import copy
import numpy as np
from scipy.stats import stats
import random
import os
import sys
import time

In [ ]:
import pickle
import re
from mlxtend.preprocessing import TransactionEncoder

#save data to loc_file
def save_data(dataset, loc_file, pre_loc_file_url = '/content/drive/My Drive/4_Proposed_Approach/'):
  loc_file = pre_loc_file_url + loc_file
  with open(loc_file, 'wb') as filehandle:
      pickle.dump(dataset, filehandle)

#load data from loc_file
def load_data(loc_file, pre_loc_file_url = '/content/drive/My Drive/4_Proposed_Approach/'):
  loc_file = pre_loc_file_url + loc_file
  print(loc_file)
  with open(loc_file, 'rb') as filehandle:
      dataset = pickle.load(filehandle)
  return dataset

def remove_subset(list_of_list):
    sets={frozenset(e) for e in list_of_list}
    us=[]
    while sets:
        e=sets.pop()
        if any(e.issubset(s) for s in sets) or any(e.issubset(s) for s in us):
            continue
        else:
            us.append(sorted(list(e)))
    return us

def convert_transaction_to_df_one_hot_vector(_transaction_dataset, sparse_format = True):
  te = TransactionEncoder()
  if(sparse_format):
    te_ary = te.fit(_transaction_dataset).transform(_transaction_dataset, sparse=True)
    return pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)
  te_ary = te.fit(_transaction_dataset).transform(_transaction_dataset)
  return pd.DataFrame(te_ary, columns=te.columns_)

def print_list_with_minK(list_of_list, min_k_item):
  count = 0
  for item in list_of_list:
    if(len(item) >= min_k_item):
      count +=1
      print(item)
  print(f"Len -origin:{len(list_of_list)} ")
  print(f"Len -filter by min_k_item={min_k_item} :{count} ")

def is_binary_device_by_df(df):
    if (df['device_id'][0:1] == 'M'
        or df['device_id'][0:1] == 'D'
        or df['device_id'][0:2] == 'L0'):
        val = bool(True)
    else:
        val = bool(False)
    return val

def is_binary_device_by_id(device_id):
    if (device_id[0:1] == 'M'
        or device_id[0:1] == 'D'
        or device_id[0:2] == 'L0'):
        val = bool(True)
    else:
        val = bool(False)
    return val

def is_actuator_by_id(device_id):
  if re.search("[LD]\\d+", device_id):
    return True
  return False

In [ ]:
from enum import Enum

class Dataset(Enum):
  HH102 = 'hh102'
  HH106 = 'hh106'
  HH108 = 'hh108'
  HH113 = 'hh113'
  HH114 = 'hh114'

class Anomaly(Enum):
  INTERFERENCE = 'interference'
  LOCATION = 'location'
  MALFUNCTION = 'malfunction'
  STUCK_AT = 'stuck_at'

class Experiments:
  MIN_SUP = 'min sup'
  PERCENTILE = 'percentile'
  DISTANCE_TYPE = 'distance type'
  COMPACT = 'compact'
  EXECUTION_TIME = 'execution time'

class TestDataService:
  BASE_TEST_FOLDER = '/content/drive/My Drive/4_Proposed_Approach/TEST_DATA/'

  def load_single_test_data(self, anomaly_type, dataset, test_data_index):
    raw_data = load_data(pre_loc_file_url=self.BASE_TEST_FOLDER, loc_file=f'{anomaly_type}/{dataset}/{dataset}_{Anomaly[anomaly_type].value}_{test_data_index}')
    return raw_data

  def load_test_dataset(self, anomaly_type, dataset, start_test_data_index, end_test_data_index):
    test_dataset = []
    for i in range(start_test_data_index, end_test_data_index + 1):
      test_dataset.append(self.load_single_test_data(anomaly_type, dataset, i))
    return test_dataset

**Detect anomaly**

In [ ]:
class ProcessedSensorEvents:
  def __init__(self, datetime, device_id, device_value, is_binary_device):
    self.datetime = datetime
    self.device_id = device_id
    self.device_value = device_value
    self.is_binary_device = is_binary_device

class TestConfig:
  def __init__(self, data_type, anomaly_type, start_test_data_index, end_test_data_index, start_time_jump, time_distance):
    self.data_type = data_type
    self.anomaly_type = anomaly_type
    self.start_test_data_index = start_test_data_index
    self.end_test_data_index = end_test_data_index
    self.start_time_jump = start_time_jump
    self.time_distance = time_distance

In [ ]:
from sklearn.neighbors import NearestNeighbors
import math
import re
from scipy.spatial import distance
from statistics import mean
from operator import itemgetter

class ProposedApproach:
  TIME_WINDOW_IN_MINUTES = 2
  transaction_dataset = []
  number_of_hours = 0*24 # ~ 4 weeks

  def __init__(self):
    self.test_data_service = TestDataService()

  def reset_state(self):
    self.transaction_dataset = []

  def get_potential_anomalies(self, anomalies_dict, amount_threshold=2):
    sorted_anomalies_dict = sorted(anomalies_dict.items(), key=lambda x:x[1], reverse=True)
    converted_dict = dict(sorted_anomalies_dict)
    potential_anomalies = []

    for i in range(amount_threshold):
      anomaly_device = list(converted_dict.keys())[i]
      if converted_dict[anomaly_device] > 1:
        potential_anomalies.append(anomaly_device)

    return potential_anomalies

  def convert_to_processed_sensor_events(self, data):
    raw_data = data.values.tolist()
    return list(map(lambda x: ProcessedSensorEvents(x[0], x[1], x[2], x[3]), raw_data))

  def get_device_info(self, data_raw):
    self.device_info = pd.DataFrame([], columns=['device_id', 'is_binary_device'])
    self.device_info['device_id'] = data_raw['device_id'].unique()
    self.device_info['is_binary_device'] = self.device_info['device_id'].apply(lambda x: data_raw[data_raw['device_id'] == x]['is_binary_device'].iloc[0])

  def get_actuators_from_rules(self, device_set):
    return list(filter(is_actuator_by_id, device_set))

  def init_state(self, dataset_name, data_raw):
    start_test_date = self.test_data_service.load_single_test_data(Anomaly.INTERFERENCE.name, dataset_name, 1).iloc[0].datetime

    recent_values = {}

    device_set = set()

    for rule_index, ru in self.rules.iterrows():
      # print(ru['itemsets'])
      for device in ru['itemsets']:
        if device not in device_set:
          recent_value = data_raw.loc[(data_raw['device_id'] == device) & (data_raw['datetime'] <= start_test_date)].iloc[-1].device_value
          recent_values[device] = recent_value
          device_set.add(device)

    self.recent_values = recent_values
    self.actuators = self.get_actuators_from_rules(list(device_set))
    print(self.recent_values)

  def get_initial_missing_devices_map(self):
    missing_device_map = {}
    for rule_index, ru in self.rules.iterrows():
      new_rule_map = {key: 0 for key in ru['itemsets']}
      missing_device_map[rule_index] = new_rule_map
    return missing_device_map

  def is_only_binaries_rule(self, itemset):
    for item in itemset:
      is_binary = self.device_info[self.device_info['device_id'] == item].iloc[0].is_binary_device
      if not is_binary:
        return False
      return True

  def execute_v3(self, data, time_jump, time_distance=4, log=True):
    copy_recent_values = {}
    for key in self.recent_values.keys():
      copy_recent_values[key] = self.recent_values[key]

    start_date = data.iloc[0].datetime + datetime.timedelta(hours=time_jump)
    # start_date = time_jump
    if time_distance != 0:
      last_datetime = start_date + datetime.timedelta(hours=time_distance)
    else:
      last_datetime = data.iloc[-1].datetime

    data_to_make_group_dev = data.loc[(data['datetime'] >= start_date) & (data['datetime'] <= last_datetime)]

    true_positive = 0
    false_positive = 0
    false_negative = 0

    # extract all actuator events
    actuator_data = data_to_make_group_dev[data_to_make_group_dev['device_id'].isin(self.actuators)]

    actuator_events = self.convert_to_processed_sensor_events(actuator_data)

    anomaly_devices = data[data['is_generated'] == True]['device_id'].unique()
    # print('Anomalous devices:', anomaly_devices)

    rule_end_time = {}
    missing_devices_map = self.get_initial_missing_devices_map()
    rule_recent_datapoint = {}

    # Assignment 3: Record time taken to detect anomalies
    anomaly_detected_time = None

    for rule_index, ru in self.rules.iterrows():
      rule_end_time[rule_index] = -1
      rule_recent_datapoint[rule_index] = []

    for event in actuator_events:
      # extract time window that centers around the actuator
      actuator_timestamp = event.datetime
      actuator = event.device_id
      window_start = actuator_timestamp - datetime.timedelta(minutes=self.TIME_WINDOW_IN_MINUTES/2)
      window_end = actuator_timestamp + datetime.timedelta(minutes=self.TIME_WINDOW_IN_MINUTES/2)

      actuator_timewindow = data.loc[(data['datetime'] > window_start) & (data['datetime'] < window_end)]

      if not actuator_timewindow.empty:
        for rule_index, ru in self.rules.iterrows():
          itemset = list(ru['itemsets'])
          # only consider rules that contains the actuator
          if actuator in itemset:
            # check overlapping rule
            rule_start = actuator_timewindow[actuator_timewindow['device_id'].isin(itemset)].iloc[0].datetime
            rule_end = actuator_timewindow[actuator_timewindow['device_id'].isin(itemset)].iloc[-1].datetime

            if rule_end_time[rule_index] == -1:
              rule_end_time[rule_index] = [rule_start, rule_end]
            else:
              if rule_start == rule_end_time[rule_index][0] and rule_end == rule_end_time[rule_index][1]:
                continue
              else:
                rule_end_time[rule_index] = [rule_start, rule_end]

            num_of_devices = 0
            datapoint = []
            threshold = ru['threshold']
            nbrs = ru['model']
            missing_devices = []

            for device in itemset:
              numRec = actuator_timewindow[actuator_timewindow['device_id'] == device].shape[0]
              value = 0

              if numRec == 0:
                # if there is no event belong to device, take the past value
                missing_devices.append(device)

                prev_device_events = data.loc[(data['device_id'] == device) & (data['datetime'] < window_start)]
                if prev_device_events.shape[0] == 0:
                  value = copy_recent_values[device]
                else:
                  value = prev_device_events.iloc[-1].device_value

              else:
                value = math.ceil(actuator_timewindow[actuator_timewindow['device_id'] == device]['device_value'].mean())
                num_of_devices += 1

              if self.device_info[self.device_info['device_id'] == device]['is_binary_device'].iloc[0] == True:
                if value > 0:
                  value = 50
                else:
                  value = 0

              datapoint.append(value)

            # if there is not enough devices to consider the rule
            if num_of_devices / len(itemset) < 0.5:
              continue
            if rule_recent_datapoint[rule_index] == datapoint and not self.is_only_binaries_rule(ru['itemsets']):
              continue

            anomaly_by_missing_devices = []
            # print(missing_devices_map)

            for device in ru['itemsets']:
              if device in missing_devices:
                missing_devices_map[rule_index][device] += 1
                if missing_devices_map[rule_index][device] >= 3:
                  anomaly_by_missing_devices.append(device)
              else:
                if missing_devices_map[rule_index][device] != 0:
                  missing_devices_map[rule_index][device] = 0

            Y = [datapoint]
            rule_recent_datapoint[rule_index] = datapoint
            dis, inc = nbrs.kneighbors(Y)

            if self.is_only_binaries_rule(ru['itemsets']):
              check_neighbor = False
              for indice in inc[0]:
                if datapoint == list(ru['input'][indice]):
                  check_neighbor = True
                  break
              if check_neighbor:
                continue

            anomalies = []
            if dis.min() > threshold:
              # test anomaly device detection with point replacement
              potential_anomalies = {key: 0 for key in list(ru['itemsets'])}

              for point_index in range(len(datapoint)):
                for indice in inc[0]:
                  neighbor_point = ru['input'][indice]
                  copy_datapoint = datapoint.copy()
                  copy_datapoint[point_index] = neighbor_point[point_index]
                  dis, inc = nbrs.kneighbors([copy_datapoint])
                  if dis.min() <= threshold:
                    potential_anomalies[list(ru['itemsets'])[point_index]] += 1

              anomalies = self.get_potential_anomalies(potential_anomalies)

            for missing_device in anomaly_by_missing_devices:
              if missing_device not in anomalies:
                anomalies.append(missing_device)

            for device in anomalies:
                if device in anomaly_devices:
                  true_positive += 1
                  if anomaly_detected_time is None:
                    anomaly_detected_time = actuator_timestamp - start_date
                else:
                  false_positive += 1

            for device in anomaly_devices:
              if device not in anomalies and device in list(ru['itemsets']):
                false_negative += 1

            if log:
              print('Anomaly devices ', anomalies)
              print("Missing devices: ", missing_devices)
              print('Anomaly at time window from',' ',window_start,' to ',window_end,' for the rule ',rule_index,' with the datapoint ',datapoint)
              print('Anomaly at time window from',' ',window_start,' to ',window_end,' for the rule ',rule_index,' with distances ', dis)
              print('Anomaly at time window from',' ',window_start,' to ',window_end,' for the rule ',rule_index,' with the itemset ',list(ru['itemsets']))
              print("Distances of anomaly: ", dis)
              print("K neighbors of anomaly: ")
              for indice in inc[0]:
                neighbor_point = ru['input'][indice]
                print(neighbor_point, ru['weight'][indice])
              print("########################")

    if true_positive + false_positive == 0:
      precision = -1
    else:
      precision = true_positive / (true_positive + false_positive) * 100
    if true_positive + false_negative == 0:
      recall = -1
    else:
      recall = true_positive / (true_positive + false_negative) * 100
    print(f"False positive: {false_positive}, total events: {data_to_make_group_dev.shape[0]}")

    return precision, recall, anomaly_detected_time

  def run_test_cases(self, data_type, anomaly_type, start_test_data_index, end_test_data_index, start_time_jump, time_distance):
    test_datas = self.test_data_service.load_test_dataset(anomaly_type, data_type, start_test_data_index, end_test_data_index)
    precisions = []
    recalls = []

    for test_data in test_datas:
      anomaly_devices = test_data[test_data['is_generated'] == True]['device_id'].unique()
      check_rule = False
      actuator = None

      for rule_index, ru in self.rules.iterrows():
        if check_rule:
          break
        if anomaly_devices[0] in list(ru['itemsets']):
          check_rule = True
          for device in list(ru['itemsets']):
            if is_actuator_by_id(device):
              actuator = device
              break

      if check_rule:
        act_event = test_data[test_data['device_id'] == actuator]
      else:
        continue

      if check_rule and act_event.shape[0] != 0:
        first_act_event = act_event.iloc[0].datetime
        datetime_distance = first_act_event - test_data.iloc[0].datetime
        start_time_jump = datetime_distance.days * 24 + math.floor(datetime_distance.seconds / 3600)
      print("Start time after: ", start_time_jump)

      start_time = time.time()
      precision, recall, _ = self.execute_v3(test_data, start_time_jump, time_distance, False)
      execution_time_in_seconds = time.time() - start_time

      print(anomaly_devices, precision, recall)

      if precision < 0:
        precision = 0
      if recall < 0:
        recall = 0
      precisions.append(precision)
      recalls.append(recall)

    avg_precison = sum(precisions) / len(precisions)
    avg_recall = sum(recalls) / len(recalls)
    return avg_precison, avg_recall

  def bar_plot(self, data, data_labels, data_groups, fig_name):
    BAR_WIDTH = 0.1
    Y_LABEL = 'y-label'
    X_axis = np.arange(len(data[0]))

    bar_per_grp = len(data_labels)
    bar_location = []
    if (bar_per_grp % 2 == 1):
        bar_location = [i for i in range(int(-bar_per_grp/2), int(bar_per_grp/2) + 1)]
    else:
        bar_location = [i for i in range(int(-bar_per_grp/2) + 1, int(bar_per_grp/2) + 1)]
        bar_location = [i - 0.5 for i in bar_location]

    for index,row in enumerate(data):
        plt.bar(X_axis + (BAR_WIDTH * bar_location[index]), row, BAR_WIDTH, label = data_labels[index])

    plt.xticks(X_axis, data_groups)
    plt.subplots_adjust(bottom=0.15)
    plt.xlabel('Dataset', fontsize=12)
    plt.ylabel('Result (%)', fontsize=12)
    plt.ylim(0, 100)
    plt.legend(bbox_to_anchor=(0, -0.13, 1, 0.1), loc="lower left", mode="expand", borderaxespad=0, ncol=len(data_labels))

    plt.rc('font', size=12)          # controls default text sizes
    plt.rc('axes', titlesize=10)     # fontsize of the axes title
    plt.rc('axes', labelsize=10)    # fontsize of the x and y labels
    plt.rc('xtick', labelsize=16)    # fontsize of the tick labels
    plt.rc('ytick', labelsize=10)    # fontsize of the tick labels
    plt.rc('legend', fontsize=27)    # legend fontsize
    plt.rc('figure', titlesize=18)  # fontsize of the figure title

    plt.savefig(fig_name)

  def line_plot(self, data, xlabels, x_axis_label, data_label, fig_name):
    x_axis = [i for i in range(1, len(xlabels) + 1)]
    fig = plt.figure(figsize=(12, 9))

    for j in range(len(data)):
      plt.plot(x_axis, data[j], label=data_label[j], marker='o')
      plt.xticks(x_axis, xlabels)

    plt.xlabel(x_axis_label, fontsize=18)
    plt.ylabel('Result (%)', fontsize=18)
    plt.ylim(50, 100)
    plt.legend(bbox_to_anchor=(0, -0.2, 1, 0.1), loc="lower left", mode="expand", borderaxespad=0, ncol=len(data_label), fontsize="20")

    plt.rc('font', size=12)          # controls default text sizes
    plt.rc('axes', titlesize=10)     # fontsize of the axes title
    plt.rc('axes', labelsize=10)    # fontsize of the x and y labels
    plt.rc('xtick', labelsize=16)    # fontsize of the tick labels
    plt.rc('ytick', labelsize=10)    # fontsize of the tick labels
    plt.rc('legend', fontsize=50)    # legend fontsize
    plt.rc('figure', titlesize=18)  # fontsize of the figure title

    plt.savefig(fig_name, bbox_inches='tight')

  def array_reshape(self, src_arr):
    length = len(src_arr)
    width = len(src_arr[0])

    res = []
    for i in range(width):
      new_row = []
      for j in range(length):
        new_row.append(src_arr[j][i])
      res.append(new_row)

    return res

  def test_execution_time(self):
    rules_list = {}
    data_labels = []

    for dataset in Dataset:
      rules_list[dataset.name] = load_data(f'Non_compact_models/min sup 80/90/mahalanobis/mahalanobis_{dataset.value}_90.model')
      data_labels.append(dataset.name)

    anomaly_type = Anomaly.INTERFERENCE.name
    execution_times = []
    event_nums = []
    test_times = []

    for dataset in Dataset:
      past_data = load_data(f'Past_data/{dataset.value}_training.data')
      data_type = dataset.value

      self.get_device_info(past_data)

      self.rules = rules_list[dataset.name]
      self.init_state(dataset.value, past_data)

      test_data = self.test_data_service.load_single_test_data(anomaly_type, data_type, 1)
      event_nums.append(test_data.shape[0])
      test_times.append(round((test_data.iloc[-1].datetime - test_data.iloc[0].datetime).seconds / 3600))

      start_time = time.time()
      precision, recall, _ = self.execute_v3(test_data, 0, 0, False)
      execution_time_in_seconds = time.time() - start_time

      execution_times.append(execution_time_in_seconds)

    execution_time_plot = plt.figure(1)
    plt.bar(data_labels, execution_times)
    plt.xlabel("Dataset")
    plt.ylabel("Execution time (seconds)")

    event_num_plot = plt.figure(2)
    plt.bar(data_labels, event_nums)
    plt.xlabel("Dataset")
    plt.ylabel("Event num")

    test_time_plot = plt.figure(3)
    plt.bar(data_labels, test_times)
    plt.xlabel("Dataset")
    plt.ylabel("Test time (hours)")

    plt.show()

  def check_anomalies_coverage(self):
    avg_test_lines = []
    anomaly_labels = []

    for anomaly in Anomaly:
      anomaly_labels.append(anomaly.name)

      avg_test_percen = []
      for dataset in Dataset:
        test_datas = self.test_data_service.load_test_dataset(anomaly.name, dataset.value, 1, 10)
        for test_data in test_datas:
          test_line = test_data[test_data['is_generated'] == True].shape[0]
          total_line = test_data.shape[0]
          avg_test_percen.append(test_line / total_line * 100)
      avg_test_lines.append(sum(avg_test_percen) / len(avg_test_percen))

    test_time_plot = plt.figure()
    plt.bar(anomaly_labels, avg_test_lines)
    plt.xlabel("Anomaly")
    plt.ylabel("Total anomalies generated compare to normal data (%)")

    plt.show()

  def plot(self, experiment_type: Experiments):
    rules_list = {}
    min_sup_list = [40, 50, 60, 70, 80, 90]
    data_labels = []
    percentile_list = [50, 60, 70, 80, 90, 99]
    distance_list = ['Mahalanobis', 'Euclidean', 'Cosine', 'Manhattan']
    compact_list = ['Non_compact', 'Compact']

    for dataset in Dataset:
      rules_list[dataset.name] = []
      data_labels.append(dataset.name)

    if experiment_type == Experiments.MIN_SUP:
      for dataset in Dataset:
        for min_sup in min_sup_list:
          rule = load_data(f'Non_compact_models/min sup {min_sup}/90/mahalanobis/mahalanobis_{dataset.value}_90.model')
          rules_list[dataset.name].append(rule)
    elif experiment_type == Experiments.PERCENTILE:
      for dataset in Dataset:
        for percentile in percentile_list:
          rule = load_data(f'Non_compact_models/min sup 80/{percentile}/mahalanobis/mahalanobis_{dataset.value}_{percentile}.model')
          rules_list[dataset.name].append(rule)
    elif experiment_type == Experiments.COMPACT:
      for dataset in Dataset:
        for compact_type in compact_list:
          rule = None
          if compact_type == 'Compact':
            rule = load_data(f'{compact_type}_models/min sup 80/90/mahalanobis/{compact_type}_mahalanobis_{dataset.value}_90.model')
          else:
            rule = load_data(f'{compact_type}_models/min sup 80/90/mahalanobis/mahalanobis_{dataset.value}_90.model')
          rules_list[dataset.name].append(rule)
    elif experiment_type == Experiments.DISTANCE_TYPE:
      for dataset in Dataset:
        for distance_type in distance_list:
          rule = load_data(f'Non_compact_models/min sup 80/90/{distance_type.lower()}/{distance_type.lower()}_{dataset.value}_90.model')
          rules_list[dataset.name].append(rule)

    anomaly_type = Anomaly.INTERFERENCE.name
    start_test_data_index = 1
    end_test_data_index = 10
    start_time_jump = 0
    time_distance = 8
    precision_results = []
    recall_results = []

    for dataset in Dataset:
      past_data = load_data(f'Past_data/{dataset.value}_training.data')
      data_type = dataset.value
      avg_precisions = []
      avg_recalls = []

      self.get_device_info(past_data)

      for rule in rules_list[dataset.name]:
        self.rules = rule
        self.init_state(dataset.value, past_data)

        avg_precision, avg_recall = self.run_test_cases(data_type, anomaly_type, start_test_data_index, end_test_data_index, start_time_jump, time_distance)
        avg_precisions.append(avg_precision)
        avg_recalls.append(avg_recall)

      precision_results.append(avg_precisions)
      recall_results.append(avg_recalls)

    precision_bar_chart_data = self.array_reshape(precision_results)
    recall_bar_chart_data = self.array_reshape(recall_results)

    if experiment_type == Experiments.MIN_SUP:
      self.line_plot(precision_results, min_sup_list, "Minimum support (%)", data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/precision_min_sup.png')
      self.line_plot(recall_results, min_sup_list, "Minimum support (%)", data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/recall_min_sup.png')
    elif experiment_type == Experiments.PERCENTILE:
      self.line_plot(precision_results, percentile_list, "Percentile value", data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/precision_percentile.png')
      self.line_plot(recall_results, percentile_list, "Percentile value", data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/recall_percentile.png')
    elif experiment_type == Experiments.COMPACT:
      self.bar_plot(precision_bar_chart_data, compact_list, data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/precision_compact.png')
      self.bar_plot(recall_bar_chart_data, compact_list, data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/recall_compact.png')
    elif experiment_type == Experiments.DISTANCE_TYPE:
      self.bar_plot(precision_bar_chart_data, distance_list, data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/precision_distance.png')
      self.bar_plot(recall_bar_chart_data, distance_list, data_labels, f'/content/drive/My Drive/4_Proposed_Approach/Test_results/recall_distance.png')


**Check data beforehand**

In [ ]:
data = load_data(loc_file='TEST_DATA/INTERFERENCE/hh102/hh102_interference_3')

/content/drive/My Drive/4_Proposed_Approach/TEST_DATA/INTERFERENCE/hh102/hh102_interference_3


In [ ]:
# data[data['is_generated'] == True]['device_id'].unique()
# data[data['device_id'] == 'LS016']
# data[data['is_generated'] == True]
# (data[data['device_id'] == 'L005'].iloc[0].datetime - data.iloc[0].datetime).seconds / 3600
# data.loc[(data['device_id'] == 'L005') & (data['device_value'].isin([0, 100]))]
# data[data['device_id'] == 'MA004'].iloc[0].datetime - data.iloc[0].datetime
data

,datetime,device_id,device_value,is_binary_device,is_generated
0,2012-07-04 05:06:36,LS018,97,False,True
1,2012-07-04 05:06:37,MA020,0,True,False
2,2012-07-04 05:06:38,MA020,1,True,False
3,2012-07-04 05:06:38,M021,1,True,False
4,2012-07-04 05:06:39,MA020,0,True,False
...,...,...,...,...,...
680733,2012-11-06 10:28:00,M019,0,True,False
680734,2012-11-06 10:28:24,M006,1,True,False
680735,2012-11-06 10:28:27,M006,0,True,False
680736,2012-11-06 10:28:52,M006,1,True,False


**Testing**

In [ ]:
proposed_approach = ProposedApproach()

In [ ]:
# init past data to learn past value
# past_data = load_data('Past_data/hh102_training.data')
# past_data = load_data(f'Past_data/{DATASET}_training.data')

# init rules
# rules = load_data('Non_compact_models/min sup 80/90/mahalanobis/mahalanobis_hh102_90.model')
# rules = load_data(f'Non_compact_models/min sup 80/90/mahalanobis/mahalanobis_{DATASET}_90.model')

In [ ]:
# proposed_approach.get_device_info(past_data)
# proposed_approach.rules = rules
# proposed_approach.init_state(Dataset.HH102.value, past_data)
# proposed_approach.init_state(DATASET_VALUE, past_data)

In [ ]:
# for i, dataset in enumerate(test_datasets):
#   print(f'------Dataset: {DATASET}  Anomaly type: {ANOMALY_TYPE}  Partition: {i + 1}------')
#   precision, recall, _ = proposed_approach.execute_v3(dataset, 0, 4)
#   print("Precision: ", precision)
#   print("Recall: ", recall)
#   print('-----------------------------------------------------------------------------')

In [ ]:
# precision, recall, _ = proposed_approach.execute_v3(data, 0, 4)
# print("Precision: ", precision)
# print("Recall: ", recall)

In [ ]:
# proposed_approach.plot(Experiments.DISTANCE_TYPE)
# proposed_approach.test_execution_time()
# proposed_approach.check_anomalies_coverage()

In [ ]:
# data_type = Dataset.HH108.value
# anomaly_type = Anomaly.INTERFERENCE.name
# start_test_data_index = 1
# end_test_data_index = 10
# start_time_jump = 0
# time_distance = 8

# proposed_approach.run_test_cases(data_type, anomaly_type, start_test_data_index, end_test_data_index, start_time_jump, time_distance)